In [27]:
import ccxt
import pandas as pd
import ta
import numpy as np
import vectorbt as vbt
import warnings
warnings.filterwarnings('ignore')

In [28]:
df = vbt.CCXTData.download('BTC/USDT', timeframe='1h', start = '6 months ago').get()

0it [00:00, ?it/s]

# ma

In [29]:
df['30ma'] = ta.trend.SMAIndicator(df['Close'], 30).sma_indicator() 

In [30]:
df['signal_ma'] = np.where(df['Close'] > df['30ma'], 1, -1)

In [31]:
df['200ma'] = ta.trend.SMAIndicator(df['Close'], 200).sma_indicator() 

In [32]:
df['signal_ma'] = np.where(df['30ma'] > df['200ma'], 1, -1)

In [34]:
df['signal_ma'] = np.where(df['30ma'].between(df['Close'] , df['200ma']) , 1 , -1)

In [35]:
ta.trend.EMAIndicator(df['Close'], 30).ema_indicator() 

Open time
2022-04-10 06:00:00+00:00             NaN
2022-04-10 07:00:00+00:00             NaN
2022-04-10 08:00:00+00:00             NaN
2022-04-10 09:00:00+00:00             NaN
2022-04-10 10:00:00+00:00             NaN
                                 ...     
2022-10-10 01:00:00+00:00    19474.547085
2022-10-10 02:00:00+00:00    19474.353724
2022-10-10 03:00:00+00:00    19472.975420
2022-10-10 04:00:00+00:00    19472.578941
2022-10-10 05:00:00+00:00    19470.767396
Freq: H, Name: ema_30, Length: 4392, dtype: float64

In [36]:
ta.trend.WMAIndicator(df['Close'], 30).wma() 

Open time
2022-04-10 06:00:00+00:00             NaN
2022-04-10 07:00:00+00:00             NaN
2022-04-10 08:00:00+00:00             NaN
2022-04-10 09:00:00+00:00             NaN
2022-04-10 10:00:00+00:00             NaN
                                 ...     
2022-10-10 01:00:00+00:00    19462.597763
2022-10-10 02:00:00+00:00    19464.526495
2022-10-10 03:00:00+00:00    19465.215935
2022-10-10 04:00:00+00:00    19466.507634
2022-10-10 05:00:00+00:00    19466.130538
Freq: H, Name: wma_30, Length: 4392, dtype: float64

In [39]:
df['cross'] = df['30ma'].vbt.crossed_above(df['200ma'], wait = 1)

In [40]:
pf = vbt.Portfolio.from_signals(df['Close'],
                                entries = (df['cross']==True),
                                exits = (df['cross']==False))
pf.stats()

Start                         2022-04-10 06:00:00+00:00
End                           2022-10-10 05:00:00+00:00
Period                                183 days 00:00:00
Start Value                                       100.0
End Value                                     99.137141
Total Return [%]                              -0.862859
Benchmark Return [%]                         -54.553337
Max Gross Exposure [%]                            100.0
Total Fees Paid                                     0.0
Max Drawdown [%]                               1.267825
Max Drawdown Duration                 110 days 08:00:00
Total Trades                                         21
Total Closed Trades                                  21
Total Open Trades                                     0
Open Trade PnL                                      0.0
Win Rate [%]                                  38.095238
Best Trade [%]                                 0.828105
Worst Trade [%]                               -0

# rsi

In [41]:
df['rsi'] = ta.momentum.RSIIndicator(df['Close']).rsi()

In [ ]:
rsi < 30, rsi > 70

In [42]:
cond_rsi = [df['rsi'] < 30, df['rsi'] > 70]
df['signal_rsi'] = np.select(cond_rsi, [1,-1], 0)

In [44]:
pf = vbt.Portfolio.from_signals(df['Close'],
                                entries = (df['signal_rsi']== -1),
                                exits = (df['signal_rsi']== 1))
pf.stats()

Start                         2022-04-10 06:00:00+00:00
End                           2022-10-10 05:00:00+00:00
Period                                183 days 00:00:00
Start Value                                       100.0
End Value                                     82.146655
Total Return [%]                             -17.853345
Benchmark Return [%]                         -54.553337
Max Gross Exposure [%]                            100.0
Total Fees Paid                                     0.0
Max Drawdown [%]                              24.889419
Max Drawdown Duration                 171 days 16:00:00
Total Trades                                         16
Total Closed Trades                                  16
Total Open Trades                                     0
Open Trade PnL                                      0.0
Win Rate [%]                                      31.25
Best Trade [%]                                 6.229747
Worst Trade [%]                               -7

# BB

In [46]:
df['bb_h'] = ta.volatility.BollingerBands(df['Close']).bollinger_hband()

In [47]:
df['bb_l'] = ta.volatility.BollingerBands(df['Close']).bollinger_lband() 

In [50]:
df['bb_h_indicator'] = ta.volatility.BollingerBands(df['Close']).bollinger_hband_indicator() 

In [48]:
df['bb_l_indicator'] = ta.volatility.BollingerBands(df['Close']).bollinger_lband_indicator() 

In [51]:
cond_bb = [ df['bb_l_indicator'] ==1, df['bb_h_indicator'] == 1 ]
df['signal_bb'] = np.select(cond_bb, [1,-1], 0)

In [52]:
df['bb_p'] = ta.volatility.BollingerBands(df['Close']).bollinger_pband() 

In [53]:
cond_bb = [df['bb_p'] < 0.1, df['bb_p'] > 0.9]
df['signal_bb'] = np.select(cond_bb, [1,-1], 0)

In [54]:
df['bb_w'] = ta.volatility.BollingerBands(df['Close']).bollinger_wband() 

In [56]:
df['signal_bb_w'] = np.where(df['bb_w'] < df['bb_w'].mean(), 1, 0)

cond_rsi = [df['rsi'] < 30, df['rsi'] > 70]
df['signal_rsi'] = np.select(cond_rsi, [1,-1], 0)

cond_go = [(df['signal_bb_w']==1) & (df['signal_rsi']==1),
          (df['signal_bb_w']==1) & (df['signal_rsi']==-1)]
df['go'] = np.select(cond_go, [1,-1], 0)

4.314459095345191

# MACD

In [57]:
df['macd_hist'] = ta.trend.MACD(df['Close']).macd_diff() 

In [58]:
df['signal_macd'] = np.where(df['macd_hist'] > 0, 1, -1)

# SO

In [59]:
df['stoch_k'] = ta.momentum.StochasticOscillator(df['Close'], df['High'], df['Low']).stoch() 
df['stoch_d'] = ta.momentum.StochasticOscillator(df['Close'], df['High'], df['Low']).stoch_signal()

In [60]:
cond_stoch = [df['stoch_k'] < 20, df['stoch_k'] > 80]
df['signal_stoch'] = np.select(cond_stoch, [1,-1], 0)

In [61]:
df['signal_stoch'] = np.where(df['stoch_k'] > df['stoch_d'] , 1 , -1)

# CLOUD

In [62]:
df['cloud'] = ta.trend.IchimokuIndicator(df['High'], df['Low']).ichimoku_a()

In [63]:
df['signal_cloud'] = np.where(df['Close'] > df['cloud'], 1, -1)

# PSAR

In [65]:
df['psar'] = ta.trend.PSARIndicator(df['High'], df['Low'], df['Close']).psar() 
df['psar_down_indicator'] = ta.trend.PSARIndicator(df['High'], df['Low'], df['Close']).psar_down_indicator()  
df['psar_up_indicator'] = ta.trend.PSARIndicator(df['High'], df['Low'], df['Close']).psar_up_indicator()  


In [66]:
df['signal_psar'] = np.where(df['Close'] > df['psar'], 1, -1)

In [67]:
cond_psar = [df['psar_up_indicator'] == 1, df['psar_down_indicator'] == 1]
df['signal_psar'] = np.select(cond_psar, [1,-1], 0)

# MFI

In [68]:
df['mfi'] = ta.volume.MFIIndicator(df['High'], df['Low'], df['Close'], df['Volume']).money_flow_index() 

In [69]:
cond_mfi = [df['mfi'] < 20, df['mfi'] > 80]
df['signal_mfi'] = np.select(cond_mfi, [1,-1], 0)

In [70]:
df['ma_mfi'] = ta.trend.SMAIndicator(df['mfi'], 3).sma_indicator()

In [71]:
df['signal_ma_mfi'] = np.where(df['mfi'] > df['ma_mfi'], 1, -1)

# PIN BAR

In [74]:
cond_pin = [ ((df['Close'] == df['High']) & ( (df['Close'] - df['Open']) < (  (df['Open'] - df['Low'])/2  )  )  )
    ,
 ( (df['Close'] == df['Low']) & ( (df['Open'] - df['Close']) < ((df['High'] - df['Open'])/2) ) )]
df['signal_pin'] = np.select(cond_pin, [1,-1], 0)


# 캔들 여러개 활용

In [80]:
df = vbt.CCXTData.download('BTC/USDT', timeframe='15m', start = '12 months ago' ).get()
# 전 캔들에서 핀바 패턴 보이면 그 다음 캔들 close 가격에 시장 진입
df['signal_pin'] = 0
for current in range(1, len(df.index)):
    previous = current -1
    if df['Close'][previous] == df['High'][previous]:
        if (df['Close'][previous] - df['Open'][previous]) < ((df['Open'][previous] - df['Low'][previous])/2):
            df['signal_pin'] = 1
    elif df['Close'][previous] == df['Low'][previous]:
        if  (df['Open'][previous] - df['Close'][previous]) < ((df['High'][previous] - df['Open'][previous])/2) :
            df['signal_pin'][current] = -1      


0it [00:00, ?it/s]

In [ ]:
df = vbt.CCXTData.download('BTC/USDT', timeframe='15m', start = '12 months ago' ).get()
df['signal_pin'] = 0
for current in range(1, len(df.index)):
    previous = current - 1
    if df.iat[previous, 3] == df.iat[previous, 1]:
        if  (df.iat[previous, 3] - df.iat[previous, 0]) < ((df.iat[previous, 0] - df.iat[previous, 2])/2) :
            df.iat[current, 5] = 1
    elif df.iat[previous, 3] == df.iat[previous, 2]:
        if  (df.iat[previous, 0] - df.iat[previous, 3]) < ((df.iat[previous, 1] - df.iat[previous, 0])/2) :
            df.iat[current, 5] = -1

In [81]:
df

,Open,High,Low,Close,Volume,signal_pin
Open time,,,,,,
2021-10-10 07:00:00+00:00,55607.80,55686.45,55512.04,55649.19,629.95301,1
2021-10-10 07:15:00+00:00,55649.18,55766.54,55411.86,55547.05,847.91125,1
2021-10-10 07:30:00+00:00,55547.05,55700.02,55464.25,55605.64,641.44695,1
2021-10-10 07:45:00+00:00,55605.63,55688.27,55523.15,55668.81,692.39433,1
2021-10-10 08:00:00+00:00,55668.81,55798.40,55598.50,55655.79,876.00036,1
...,...,...,...,...,...,...
2022-10-10 05:45:00+00:00,19404.20,19417.55,19393.53,19411.46,1229.32554,1
2022-10-10 06:00:00+00:00,19410.38,19437.12,19375.12,19427.03,2009.88369,1
2022-10-10 06:15:00+00:00,19427.03,19437.54,19418.16,19422.55,1118.91379,1


# Heikin-Ashi

In [ ]:
df = vbt.CCXTData.download('BTC/USDT', timeframe='15m', start = '12 months ago' ).get()
df_original = df.copy()
df['Close'] =  round(((df['Open'] + df['High'] + df['Low'] + df['Close'])/4),2)
for i in range(len(df)):
    if i == 0:
        df.iat[0,0] = round(((df['Open'].iloc[0] + df['Close'].iloc[0])/2),2)
    else:
        df.iat[i,0] = round(((df.iat[i-1,0] + df.iat[i-1,3])/2),2)
df['High'] = df.loc[:,['Open', 'Close']].join(df['High']).max(axis=1)
df['Low'] = df.loc[:,['Open', 'Close']].join(df['Low']).min(axis=1)

In [84]:
df['mfi'] = ta.volume.MFIIndicator(df['High'], df['Low'], df['Close'], df['Volume']).money_flow_index() 

In [85]:
cond_mfi = [df['mfi'] < 20, df['mfi'] > 80]
df['signal_mfi'] = np.select(cond_mfi, [1,-1], 0)

In [86]:
df_original['signal_mfi'] = df['signal_mfi'] 

In [ ]:
df_original